# Fairness Postprocessing Tutorial (Regression)

This tutorial demonstrates how to apply the **fairness postprocessing** method in PipeGenie for a regression task.

Since the sensitive feature is continuous, it is first divided into four groups using the helper function `prepare_data()`. Fairness mitigation is then applied **after** the model has been trained using Fairlearn's **GridSearch** algorithm.

The model is finally evaluated using both predictive performance and fairness metrics.

## 1. Load and Prepare the Dataset

We begin by loading the **Communities and Crime** dataset and defining:

- **X**: input features.
- **y**: `ViolentCrimesPerPop`, the continuous target variable.
- **A**: `racepctblack`, used as the sensitive feature.

The helper function `prepare_data()` converts the continuous sensitive feature into four quantile-based groups (Q1–Q4), making it suitable for fairness mitigation and evaluation.

The dataset is then split into training and testing sets.

In [1]:
import pandas as pd
from pipegenie.fairness import prepare_data

df = pd.read_csv("../datasets/communities.data", header=None)
    
df = df.rename(columns={7: "racepctblack", 127: "ViolentCrimesPerPop"})

y = df["ViolentCrimesPerPop"]

A = prepare_data(df["racepctblack"],n_groups=4,labels=["Q1", "Q2", "Q3", "Q4"])

X = df.drop(columns=["ViolentCrimesPerPop","racepctblack"])
X = pd.get_dummies(X, drop_first=True)

In [2]:
from pipegenie.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42)
A_train, A_test, _, _ = train_test_split(A,y,test_size=0.25,random_state=42)

## 2. Create the PipeGenie Regressor

Next, we initialize a `PipegenieRegressor` using the regression grammar.

The grammar defines the search space explored by PipeGenie to automatically generate regression pipelines.

In [3]:
grammar_file = "../../grammars/sample-grammar-regression.xml"

from pipegenie.regression import PipegenieRegressor

model = PipegenieRegressor(
grammar=grammar_file,
generations=5,
pop_size=50,
elite_size=5,
maximization=False,
n_jobs=5,
seed=9,
outdir="samples/regression_0",
)

## 3. Train the Model

The PipeGenie regressor is trained on the original training data.

At this stage, no fairness mitigation has been applied.

In [4]:
model.fit(X_train,y_train)

                                       fitness                                            size                                        fitness_elite                                      size_elite              
                 ----------------------------------------------------    --------------------------------------    ----------------------------------------------------    --------------------------------------
gen    nevals    min           max           avg           std           min    max    avg           std           min           max           avg           std           min    max    avg           std       
0      50        0.13383       2.2303e+81    5.5757e+79    3.482e+80     1      6      2.88          1.3363        0.13383       0.13722       0.13595       0.0011595     2      4      2.6           0.8       
1      34        0.13383       0.21485       0.16143       0.022852      1      6      2.88          1.194         0.13383       0.13655       0.13527       0.0

## 4. Apply Fairness Postprocessing

Once the ensemble has been trained, fairness mitigation is applied to each pipeline using Fairlearn's **GridSearch** algorithm.

This step generates multiple mitigated estimators that aim to reduce disparities between the sensitive groups while maintaining predictive performance.

In [5]:
from pipegenie.fairness import fairness_postprocessing

fair_estimators = fairness_postprocessing(ensemble=model.ensemble, X_train=X_train, y_train=y_train, sensitive_features=A_train)

## 5. Select the Best Mitigated Estimator

Several mitigated estimators are produced during postprocessing.

The helper function `select_best_mitigator()` evaluates each estimator on a validation set and returns the one with the lowest Root Mean Squared Error (RMSE).

In [6]:
from pipegenie.fairness import select_best_mitigator

best_name, best_mitigator, best_rmse = select_best_mitigator(fair_estimators, X_test, y_test)


## 6. Generate Predictions

Predictions are generated using the selected mitigated estimator.

These predictions are then used to evaluate both model performance and fairness.

In [7]:
    
y_pred = best_mitigator.predict(X_test)

## 7. Evaluate Fairness

The fairness report summarizes:

- Overall model performance.
- Performance for each sensitive group.

This helps assess whether the postprocessing method has improved fairness across the selected groups.

In [8]:
from pipegenie.fairness import fairness_report
from pipegenie.metrics import root_mean_squared_error

# =====================================================
# FAIRNESS REPORT
# =====================================================

report = fairness_report(
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=A_test,
    metric=root_mean_squared_error
)

print("\n==============================")
print("OVERALL PERFORMANCE")
print("==============================")

print(report["overall"])

print("\n==============================")
print("PERFORMANCE BY GROUP")
print("==============================")

print(report["by_group"])


OVERALL PERFORMANCE
0.14135522773124948

PERFORMANCE BY GROUP
sensitive_feature_0
Q1    0.106233
Q2    0.115390
Q3    0.136042
Q4    0.191234
Name: root_mean_squared_error, dtype: float64


## 8. Evaluate Model Performance

Finally, we evaluate the selected mitigated estimator using:

- **Model Score (R²):** Measures how well the model explains the variance in the target variable.
- **Root Mean Squared Error (RMSE):** Measures the average prediction error. Lower values indicate better predictive performance.

These metrics complement the fairness report by providing an overall assessment of the regression model.

In [9]:

# =====================================================
# STANDARD METRICS
# =====================================================

print("\n==============================")
print("ROOT MEAN SQUARED ERROR")
print("==============================")

print(
    root_mean_squared_error(
        y_test,
        y_pred
    )
)

print("\n==============================")
print("MODEL SCORE (R²)")
print("==============================")

print(
    model.score(
        X_test,
        y_test
    )
)


ROOT MEAN SQUARED ERROR
0.14135522773124948

MODEL SCORE (R²)
0.6427980606996906


## Conclusion

In this tutorial, we:

- Prepared the sensitive feature for regression.
- Trained a PipeGenie regressor.
- Applied fairness postprocessing using Fairlearn's GridSearch algorithm.
- Selected the best mitigated estimator.
- Generated predictions.
- Evaluated both predictive performance and fairness.

This workflow demonstrates how fairness can be improved **after** training while preserving the original training data.